In [3]:
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [7]:
def Preprocess (txt):
    txt = txt.lower()
    txt = re.sub(r'([?!.,/\\])', r'\1 ', txt)
    txt = re.sub(r' +', r" ", txt)
    txt = re.sub(r'[^a-zA-Z?!.,/\\]', r' ', txt)
    txt = txt.strip()
    return txt

print(Preprocess("Welcome to       Deep Learning Universe,Mikhail Cychael!"))

welcome to deep learning universe, mikhail cychael!


In [ ]:
def Load (path, example):
    with open(path, 'r') as f:
        data = f.read()
    data = Preprocess(data)
    data = data.split('\n')
    data = [d for d in data if len(d) > 0]
    data = [d.split('\t') for d in data]
    data = [(d[0], d[1]) for d in data if len(d) == 2]
    data = [(d[0], d[1]) for d in data if len(d[0]) > 0 and len(d[1]) > 0]
    data = [(d[0], d[1]) for d in data if len(d[0].split()) <= example and len(d[1].split()) <= example]
    return data

In [ ]:
def Load (path, sample):
    with open(path, 'r', encoding='utf-8') as f:
        data = f.read().strip().split('\n')
    data = [d.split('\t') for d in data[:sample]]
    input = [Preprocess(d[0]) for d in data]
    target = [Preprocess(d[1]) for d in data]
    return input, target

PATH = "D:/Practicum Binus/Deep Learning/Session 4/ind/ind.txt"
input, target = Load(PATH, 10)

print("Input: ", input)
print("Target: ", target)

Input:  ['hi.', 'run!', 'run.', 'who?', 'wow!', 'wow!', 'fire!', 'fire!', 'fire!', 'help!', 'jump!', 'jump.', 'stop!', 'stop!', 'wait!', 'wait.', 'hello.', 'hurry!', 'hurry!', 'hurry!', 'i see.', 'i see.', 'i see.', 'i see.', 'i see.', 'i see.', 'i won!', 'i won.', 'oh no!', 'relax.', 'relax.', 'shoot!', 'smile.', 'attack!', 'cheers!', 'freeze!', 'get up.', 'got it!', 'got it?', 'he ran.', 'i fell.', 'i know.', 'i know.', 'i lied.', 'i lied.', 'i wept.', 'i m   .', 'i m ok.', 'listen.', 'listen.', 'no way!', 'no way!', 'really?', 'really?', 'thanks!', 'thanks.', 'thanks.', 'we try.', 'why me?', 'ask tom.', 'awesome!', 'awesome.', 'come in.', 'come on!', 'come on!', 'get out!', 'go away!', 'goodbye!', 'goodbye.', 'he came.', 'he left.', 'he runs.', 'help me!', 'help me.', 'hold it!', 'hold on.', 'i agree.', 'i agree.', 'i cried.', 'i smoke.', 'i smoke.', 'i m sad.', 'i m sad.', 'it s ok.', 'it s me!', 'it s me.', 'kiss me.', 'kiss me.', 'me, too.', 'me, too.', 'perfect!', 'see you!', 's

In [ ]:
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
tokenizer.fit_on_texts(input + target)

input_seq = tokenizer.texts_to_sequences(input)
target_seq = tokenizer.texts_to_sequences(target)

maxlen_input = max(len(seq) for seq in input_seq)
maxlen_target = max(len(seq) for seq in target_seq)

vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size: ", vocab_size)
print(f"Input Tensor\n {input_seq}\n")
print(f"Target Tensor\n {target_seq}\n")
print(f"Tokenizer Word Index\n {tokenizer.word_index}\n")

Vocabulary Size:  16354
Input Tensor [[4448], [8570], [5761], [4449], [3113], [3113], [3638], [3638], [3638], [8571], [8572], [8573], [3639], [3639], [8574], [2693], [8575], [3640], [3640], [3640], [2, 1182], [2, 1182], [2, 1182], [2, 1182], [2, 1182], [2, 1182], [2, 8576], [2, 4450], [8577, 8578], [5762], [5762], [8579], [1936], [8580], [8581], [8582], [136, 477], [196, 2397], [196, 342], [17, 4451], [2, 4452], [2, 815], [2, 815], [2, 2135], [2, 2135], [2, 5763], [2, 36, 166], [2, 36, 1937], [3641], [3641], [101, 5764], [101, 5764], [4453], [4453], [8583], [3114], [3114], [33, 3115], [115, 493], [466, 45], [8584], [5765], [163, 1251], [163, 5766], [163, 5766], [136, 2398], [85, 8585], [8586], [3642], [17, 3116], [17, 998], [17, 8587], [257, 3117], [257, 95], [1541, 2397], [1541, 849], [2, 2399], [2, 2399], [2, 5767], [2, 1938], [2, 1938], [2, 36, 2694], [2, 36, 2694], [29, 14, 1937], [29, 14, 3117], [29, 14, 95], [1768, 95], [1768, 95], [816, 540], [816, 540], [8588], [141, 3643], [67

In [ ]:
input_train, input_test, target_train, target_test = train_test_split(input_seq, target_seq, test_size=0.2, random_state=42)

BUFFER_SIZE = len(input_train)
BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices((input_train, target_train)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

for sample in dataset.take(1):
    input_sample, target_sample = sample
    print("Input Sample: ", input_sample)
    print("Target Sample: ", target_sample)

ValueError: Can't convert non-rectangular Python sequence to Tensor.

In [ ]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)
        return output, state

    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.enc_units))